<a href="https://colab.research.google.com/github/Suhrobjonibodullayev/Mini-projects/blob/main/03_Stock_prediction/Stock_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [170]:
!pip install yfinance -q
!pip install xgboost -q

In [171]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression,  Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier, VotingClassifier, ExtraTreesClassifier, BaggingClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)



In [172]:

# Apple historical data
df = yf.download(
    "AAPL",
    start="2010-01-01",
    end="2026-09-01",
    auto_adjust=True,
    multi_level_index=False
)

df.head()

[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2010-01-04,6.400960,6.415616,6.352207,6.383612,493729600
2010-01-05,6.412028,6.448219,6.378230,6.418608,601904800
2010-01-06,6.310036,6.437451,6.303456,6.412028,552160000
2010-01-07,6.298369,6.340842,6.252608,6.333364,477131200
2010-01-08,6.340245,6.340843,6.252909,6.289997,447610800


In [173]:
df = df.reset_index()

print(df.shape)
print(df.columns)
df.head()

(4190, 6)
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


,Date,Close,High,Low,Open,Volume
0,2010-01-04,6.400960,6.415616,6.352207,6.383612,493729600
1,2010-01-05,6.412028,6.448219,6.378230,6.418608,601904800
2,2010-01-06,6.310036,6.437451,6.303456,6.412028,552160000
3,2010-01-07,6.298369,6.340842,6.252608,6.333364,477131200
4,2010-01-08,6.340245,6.340843,6.252909,6.289997,447610800


In [174]:
df.columns = df.columns.get_level_values(0)

In [175]:
df = df.reset_index()

## EDA

In [176]:
df.info() # umumiy malumotlar , ustunlar turi, ustunlarda qiymatlar to'liqligi

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4190 entries, 0 to 4189
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   index   4190 non-null   int64         
 1   Date    4190 non-null   datetime64[ns]
 2   Close   4190 non-null   float64       
 3   High    4190 non-null   float64       
 4   Low     4190 non-null   float64       
 5   Open    4190 non-null   float64       
 6   Volume  4190 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(2)
memory usage: 229.3 KB


In [177]:
df.dtypes.value_counts() # har bir malumot turi nechtaligi

,count
float64,4
int64,2
datetime64[ns],1


In [178]:
df.select_dtypes(include="object").nunique()

,0


In [179]:
df.isnull().sum().sort_values(ascending=False).head(3) # nan qiymatlar ko'rish

,0
index,0
Date,0
Close,0


In [180]:
df.duplicated().sum() # duplicatlarni ko'rish

df[df.duplicated(keep=False)].head(3) # duplicated qatorlar

,index,Date,Close,High,Low,Open,Volume


In [181]:
df.describe() # numeric ustunlar statistikasi , mean min, max, std ...

,index,Date,Close,High,Low,Open,Volume
count,4190.000000,4190,4190.000000,4190.000000,4190.000000,4190.000000,4.190000e+03
mean,2094.500000,2018-04-30 04:44:33.794749440,85.490903,86.343987,84.557421,85.415502,2.123100e+08
min,0.000000,2010-01-04 00:00:00,5.744147,5.862287,5.690307,5.753716,1.791060e+07
25%,1047.250000,2014-03-04 06:00:00,18.802992,18.978466,18.625253,18.733673,7.371210e+07
50%,2094.500000,2018-04-30 12:00:00,40.683533,40.962233,40.401129,40.701494,1.260612e+08
75%,3141.750000,2022-06-27 18:00:00,149.442707,151.378398,147.702003,149.457043,2.795568e+08
max,4189.000000,2026-08-31 00:00:00,339.786926,344.273097,337.059318,339.736982,1.880998e+09
std,1209.693143,NaN,83.844891,84.676578,82.922106,83.744754,2.137767e+08


In [182]:
df["Target"] = (
    df["Close"].shift(-1) > df["Close"]
).astype(int)

# Oxirgi qatorning ertangi ma'lumoti yo'q
df = df.iloc[:-1].copy()

In [183]:
print(df["Target"].value_counts())

Target
1    2223
0    1966
Name: count, dtype: int64


In [184]:
# =========================
# 1. MOMENTUM
# =========================

df["Return_1D"] = df["Close"].pct_change()
df["Return_5D"] = df["Close"].pct_change(5)
df["Return_20D"] = df["Close"].pct_change(20)


# =========================
# 2. TREND
# =========================

df["SMA_20"] = df["Close"].rolling(20).mean()
df["SMA_50"] = df["Close"].rolling(50).mean()
df["SMA_200"] = df["Close"].rolling(200).mean()

df["EMA_12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["EMA_26"] = df["Close"].ewm(span=26, adjust=False).mean()

# Narxning trendga nisbati
df["Close_SMA20"] = df["Close"] / df["SMA_20"] - 1
df["Close_SMA50"] = df["Close"] / df["SMA_50"] - 1
df["Close_SMA200"] = df["Close"] / df["SMA_200"] - 1


# =========================
# 3. RSI
# =========================

delta = df["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = 100 - (100 / (1 + rs))


# =========================
# 4. MACD
# =========================

df["MACD"] = df["EMA_12"] - df["EMA_26"]

df["MACD_Signal"] = (
    df["MACD"].ewm(span=9, adjust=False).mean()
)

df["MACD_Hist"] = (
    df["MACD"] - df["MACD_Signal"]
)


# =========================
# 5. VOLATILITY
# =========================

df["Volatility_20"] = (
    df["Return_1D"].rolling(20).std()
)


# =========================
# 6. VOLUME
# =========================

df["Volume_SMA20"] = df["Volume"].rolling(20).mean()

df["Volume_Ratio"] = (
    df["Volume"] / df["Volume_SMA20"]
)


# =========================
# 7. PRICE ACTION
# =========================

df["High_Low_Range"] = (
    (df["High"] - df["Low"]) / df["Close"]
)

df["Open_Close_Range"] = (
    (df["Close"] - df["Open"]) / df["Open"]
)


# NaNlarni olib tashlash
df = df.dropna().copy()

print(df.shape)

(3990, 28)


In [185]:
features = [
    # Price & Volume
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",

    # Momentum
    "Return_1D",
    "Return_5D",
    "Return_20D",

    # Trend
    "Close_SMA20",
    "Close_SMA50",
    "Close_SMA200",

    # MACD
    "MACD",
    "MACD_Signal",
    "MACD_Hist",

    # RSI
    "RSI",

    # Volatility
    "Volatility_20",

    # Volume
    "Volume_Ratio",

    # Price action
    "High_Low_Range",
    "Open_Close_Range"
]

X = df[features]
y = df["Target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3990, 19)
y shape: (3990,)


In [186]:
train = df[df["Date"] < "2022-01-01"].copy()

val = df[
    (df["Date"] >= "2022-01-01") &
    (df["Date"] < "2024-01-01")
].copy()

test = df[df["Date"] >= "2024-01-01"].copy()

X_train = train[features]
y_train = train["Target"]

X_val = val[features]
y_val = val["Target"]

X_test = test[features]
y_test = test["Target"]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (2822, 19)
Validation: (501, 19)
Test: (667, 19)


In [187]:
tscv = TimeSeriesSplit(n_splits=5)

print(tscv)

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


In [188]:
lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

In [189]:
lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

In [190]:
lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_params,
    cv=tscv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=Pipeline(steps=[('imputer',
                                        SimpleImputer(strategy='median')),
                                       ('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(max_iter=2000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'model__C': [0.01, 0.1, 1, 10, 100],
                         'model__class_weight': [None, 'balanced']},
             scoring='roc_auc', verbose=1)

In [191]:
print("Best parameters:")
print(lr_grid.best_params_)

print("\nBest CV ROC-AUC:")
print(lr_grid.best_score_)

Best parameters:
{'model__C': 10, 'model__class_weight': 'balanced'}

Best CV ROC-AUC:
0.521692473862608


In [192]:
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

In [193]:
rf_params = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [5, 10, 20, None],
    "model__min_samples_leaf": [1, 3, 5],
    "model__max_features": ["sqrt", "log2"]
}

In [194]:
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_params,
    cv=tscv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=Pipeline(steps=[('imputer',
                                        SimpleImputer(strategy='median')),
                                       ('model',
                                        RandomForestClassifier(n_jobs=-1,
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [5, 10, 20, None],
                         'model__max_features': ['sqrt', 'log2'],
                         'model__min_samples_leaf': [1, 3, 5],
                         'model__n_estimators': [200, 400]},
             scoring='roc_auc', verbose=1)

In [195]:
print("Best parameters:")
print(rf_grid.best_params_)

print("\nBest CV ROC-AUC:")
print(rf_grid.best_score_)

Best parameters:
{'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 400}

Best CV ROC-AUC:
0.5129508884020891


In [196]:
xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

In [197]:
xgb_params = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [2, 3, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

In [198]:
xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_params,
    cv=tscv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=Pipeline(steps=[('imputer',
                                        SimpleImputer(strategy='median')),
                                       ('model',
                                        XGBClassifier(base_score=None,
                                                      booster=None,
                                                      callbacks=None,
                                                      colsample_bylevel=None,
                                                      colsample_bynode=None,
                                                      colsample_bytree=None,
                                                      device=None,
                                                      early_stopping_rounds=None,
                                                      enable_categorical=Tr...
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=-1,
                                                      num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'model__colsample_bytree': [0.8, 1.0],
                         'model__learning_rate': [0.03, 0.05, 0.1],
                         'model__max_depth': [2, 3, 5],
                         'model__n_estimators': [100, 200, 300],
                         'model__subsample': [0.8, 1.0]},
             scoring='roc_auc', verbose=1)

In [199]:
print("Best parameters:")
print(xgb_grid.best_params_)

print("\nBest CV ROC-AUC:")
print(xgb_grid.best_score_)

Best parameters:
{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 100, 'model__subsample': 0.8}

Best CV ROC-AUC:
0.5144572319047312


In [200]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "CV ROC-AUC": [
        lr_grid.best_score_,
        rf_grid.best_score_,
        xgb_grid.best_score_
    ]
})

results.sort_values(
    "CV ROC-AUC",
    ascending=False
)

,Model,CV ROC-AUC
0,Logistic Regression,0.521692
2,XGBoost,0.514457
1,Random Forest,0.512951


In [201]:
models = {
    "Logistic Regression": lr_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_,
    "XGBoost": xgb_grid.best_estimator_
}

for name, model in models.items():

    pred = model.predict(X_val)
    prob = model.predict_proba(X_val)[:, 1]

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Accuracy :", accuracy_score(y_val, pred))
    print("Precision:", precision_score(y_val, pred))
    print("Recall   :", recall_score(y_val, pred))
    print("F1       :", f1_score(y_val, pred))
    print("ROC-AUC  :", roc_auc_score(y_val, prob))


Logistic Regression
Accuracy : 0.48303393213572854
Precision: 0.4967948717948718
Recall   : 0.603112840466926
F1       : 0.5448154657293497
ROC-AUC  : 0.47761051221534734

Random Forest
Accuracy : 0.5449101796407185
Precision: 0.5390835579514824
Recall   : 0.7782101167315175
F1       : 0.6369426751592356
ROC-AUC  : 0.534046692607004

XGBoost
Accuracy : 0.5409181636726547
Precision: 0.542319749216301
Recall   : 0.6731517509727627
F1       : 0.6006944444444444
ROC-AUC  : 0.5404414109842445


In [202]:
for name, model in models.items():

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Accuracy :", round(accuracy_score(y_test, pred), 4))
    print("Precision:", round(precision_score(y_test, pred), 4))
    print("Recall   :", round(recall_score(y_test, pred), 4))
    print("F1       :", round(f1_score(y_test, pred), 4))
    print("ROC-AUC  :", round(roc_auc_score(y_test, prob), 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, pred))


Logistic Regression
Accuracy : 0.4873
Precision: 0.5236
Recall   : 0.5845
F1       : 0.5524
ROC-AUC  : 0.4575

Confusion Matrix:
[[114 192]
 [150 211]]

Random Forest
Accuracy : 0.5217
Precision: 0.55
Recall   : 0.6399
F1       : 0.5915
ROC-AUC  : 0.5133

Confusion Matrix:
[[117 189]
 [130 231]]

XGBoost
Accuracy : 0.5007
Precision: 0.5341
Recall   : 0.6066
F1       : 0.5681
ROC-AUC  : 0.5128

Confusion Matrix:
[[115 191]
 [142 219]]
